#Ingest circuit.csv file
    1.Read the file Using Spark dataframe reader API 
    2. Add MetaData Columns 
    .Source File 
    .Ingestion Timestamp
    3. Write bronze delta table 

In [0]:
%run ../00.common/00.Common_source_file

In [0]:
%run ../00.common/01.Bronze_helper

In [0]:
source_file = f"{landing_file}/circuits.csv"
table_name = f"{catalog_name}.{bronze_name}.circuits"

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
circuits_schema =  StructType([
    StructField("circuitId", StringType()),
    StructField("url", StringType()),
    StructField("circuitName", StringType()), 
    StructField("lat", DoubleType()),
    StructField("long", DoubleType()),
    StructField("locality", StringType()),
    StructField("country", StringType())

])


In [0]:
%python
circuits_df = (
    spark.read
        .format("csv")
        .option("header", True)
        .option("mode", "FAILFAST")
        .schema(circuits_schema)
        ).load(source_file)

In [0]:
%python
display(circuits_df)

#Here are the DataFrames in the session:
#- Spark DataFrame 'dataframe_cluster', columns = ['_c0': string, '_c1': string, '_c2': string, '_c3

In [0]:
%python

circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
%python
display(circuits_final_df)

## Write to Bronze delta table 


In [0]:
%python
(
    circuits_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)

)

In [0]:
%sql
SELECT *
FROM formula1.bronze.circuits